In [ ]:
#!rm -rf /content/job-postings-fraud
%cd /content

# Clone your repo
!git clone https://github.com/tommygarner/job-postings-fraud.git
%cd job-postings-fraud

# Install dependencies
!pip install -r requirements_app.txt
!pip install lime shap captum keras_preprocessing

import os

RESULTS_DIR = "results/interpretability"
os.makedirs(RESULTS_DIR, exist_ok=True)

/content
Cloning into 'job-postings-fraud'...
remote: Enumerating objects: 430, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 430 (delta 40), reused 59 (delta 23), pack-reused 349 (from 1)
Receiving objects: 100% (430/430), 67.41 MiB | 11.41 MiB/s, done.
Resolving deltas: 100% (194/194), done.
Updating files: 100% (67/67), done.
/content/job-postings-fraud
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 44.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error 

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Same preprocessing as training notebook
DATA_PATH = Path("data/raw/fake_job_postings.csv")
df = pd.read_csv(DATA_PATH)

def combine_text_fields(row, fields=(
    "title", "location", "company_profile", "description",
    "requirements", "benefits", "required_experience",
    "required_education", "industry", "function",
)):
    parts = []
    for f in fields:
        if pd.notna(row[f]):
            parts.append(str(row[f]))
    return " ".join(parts) if parts else "unknown job"

df["combined_text"] = df.apply(combine_text_fields, axis=1)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text):
        return ""
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

df["text_processed"] = df["combined_text"].apply(preprocess_text)

X_text_all = df["text_processed"].values
y_all = df["fraudulent"].values.astype("int32")

# Recreate the same 70/15/15 stratified split
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text_all,
    y_all,
    test_size=0.30,
    random_state=42,
    stratify=y_all,
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [10]:
from pathlib import Path
import joblib
import pickle
import types
import sys
import tensorflow as tf
from keras.layers import TFSMLayer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODELS_DIR = Path("models")

# --- Shim so old pickled tokenizer can be loaded ---
from tensorflow.keras.preprocessing.text import Tokenizer

keras_module = types.ModuleType("keras")
keras_preprocessing = types.ModuleType("keras.preprocessing")
keras_preprocessing_text = types.ModuleType("keras.preprocessing.text")
keras_preprocessing_text.Tokenizer = Tokenizer

sys.modules["keras"] = keras_module
sys.modules["keras.preprocessing"] = keras_preprocessing
sys.modules["keras.preprocessing.text"] = keras_preprocessing_text

# 1. Naive Bayes pipeline
nb_model = joblib.load(MODELS_DIR / "naive_bayes_model.pkl")

# 2. LSTM tokenizer + SavedModel (TFSMLayer with Keras 3)
with open(MODELS_DIR / "tokenizer.pkl", "rb") as handle:
    lstm_tokenizer = pickle.load(handle)

lstm_layer = TFSMLayer(str(MODELS_DIR / "lstm_savedmodel"),
                       call_endpoint="serving_default")

def lstm_predict_proba(x_pad):
    x_tf = tf.convert_to_tensor(x_pad, dtype=tf.float32)
    out = lstm_layer(x_tf)
    if isinstance(out, dict):
        y = out["dense_1"]
    else:
        y = out
    return y.numpy().ravel()

# 3. MiniLM tokenizer + model
minilm_dir = MODELS_DIR / "model_miniLM_final"
minilm_tokenizer = AutoTokenizer.from_pretrained(str(minilm_dir))
minilm_model = AutoModelForSequenceClassification.from_pretrained(str(minilm_dir))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
minilm_model.to(device).eval()


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 384, padding_idx=0)
      (position_embeddings): Embedding(512, 384)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm((384,), eps=1e

# NB

In [ ]:
from joblib import load
from scipy.sparse import hstack, csr_matrix
import numpy as np

# 1. Load vectorizer
vectorizer = load("models/vectorizer.pkl")

# 2. Rebuild splits directly on df (same 70/15/15 as LSTM notebook)
df_train, df_temp = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["fraudulent"]
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, random_state=42, stratify=df_temp["fraudulent"]
)

# 3. Text features for NB
X_text_test_nb = df_test["text_processed"].values
text_features = vectorizer.transform(X_text_test_nb)

# 4. Numeric features (match app.py)
telecommuting = df_test["telecommuting"].fillna(0).astype(int).values
has_logo = df_test["has_company_logo"].fillna(0).astype(int).values
has_questions = df_test["has_questions"].fillna(0).astype(int).values
location_fraud_ratio = np.full_like(telecommuting, 0.05, dtype=float)
char_count = df_test["combined_text"].fillna("").str.len().values

numeric = np.vstack(
    [telecommuting, has_logo, has_questions, location_fraud_ratio, char_count]
).T
numeric_sparse = csr_matrix(numeric)

# 5. Final X_test and y_test for NB / MiniLM / ensemble
X_test = hstack([text_features, numeric_sparse])
y_test = df_test["fraudulent"].values.astype("int32")

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape, "fraud rate:", y_test.mean())


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


X_test shape: (2682, 5005)
y_test shape: (2682,) fraud rate: 0.048471290082028336


In [17]:
X_text_val_nb = df_val["text_processed"].values
y_val = df_val["fraudulent"].values.astype("int32")

X_text_test_nb = df_test["text_processed"].values
y_test = df_test["fraudulent"].values.astype("int32")

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)
    return acc, f1, prec, rec, auc

# Use the NB model that was trained on [TF‑IDF | numeric] features
nb_probs = nb_model.predict_proba(X_test)[:, 1]

nb_acc, nb_f1, nb_prec, nb_rec, nb_auc = compute_binary_metrics(
    y_test, nb_probs, threshold=0.5
)

print("NB Accuracy :", nb_acc)
print("NB F1-Score :", nb_f1)
print("NB Precision:", nb_prec)
print("NB Recall   :", nb_rec)
print("NB ROC-AUC  :", nb_auc)

NB Accuracy : 0.959731543624161
NB F1-Score : 0.39325842696629215
NB Precision: 0.7291666666666666
NB Recall   : 0.2692307692307692
NB ROC-AUC  : 0.7937726067036412


# LSTM

In [13]:
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.layers import TFSMLayer

# Load tokenizer
with open("models/tokenizer.pkl", "rb") as handle:
    lstm_tokenizer = pickle.load(handle)

# Wrap SavedModel (we know signature is 'serving_default' and output key is 'dense_1')
lstm_layer = TFSMLayer("models/lstm_savedmodel", call_endpoint="serving_default")

def lstm_predict_proba(x_pad):
    x_tf = tf.convert_to_tensor(x_pad, dtype=tf.float32)
    out = lstm_layer(x_tf)
    if isinstance(out, dict):
        y = out["dense_1"]
    else:
        y = out
    return y.numpy().ravel()

MAX_SEQ_LEN = 256

# Text → sequences → pad
lstm_sequences = lstm_tokenizer.texts_to_sequences(X_test_text)
lstm_padded = pad_sequences(
    lstm_sequences,
    maxlen=MAX_SEQ_LEN,
    padding="post",
    truncating="post"
)

# Probabilities + metrics
lstm_probs = lstm_predict_proba(lstm_padded)

lstm_acc, lstm_f1, lstm_prec, lstm_rec, lstm_auc = compute_binary_metrics(
    y_test,
    lstm_probs,
    threshold=0.5
)

print(f"LSTM Accuracy:", lstm_acc)
print(f"LSTM F1-Score:", lstm_f1)
print(f"LSTM Precision:", lstm_prec)
print(f"LSTM Recall   :", lstm_rec)
print(f"LSTM ROC-AUC  :", lstm_auc)

LSTM Accuracy: 0.9839671886651753
LSTM F1-Score: 0.8313725490196079
LSTM Precision: 0.848
LSTM Recall   : 0.8153846153846154
LSTM ROC-AUC  : 0.9848655654690137


# MINILM

In [20]:
!pip install tqdm
from tqdm.auto import tqdm


In [23]:
def get_minilm_probs(text_array):
    texts = list(text_array)
    batch_size = 64
    all_probs = []
    fraud_idx = minilm_model.config.label2id["fraud"]

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="MiniLM batches"):
            batch_texts = texts[i:i+batch_size]
            enc = minilm_tokenizer(
                batch_texts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=256,
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = minilm_model(**enc)
            probs = torch.softmax(outputs.logits, dim=1)[:, fraud_idx]
            all_probs.extend(probs.cpu().numpy().tolist())

    return np.array(all_probs)


In [24]:
minilm_val_probs_raw = get_minilm_probs(X_text_val_nb)
minilm_test_probs_raw = get_minilm_probs(X_text_test_nb)

MiniLM batches:   0%|          | 0/42 [00:00<?, ?it/s]

MiniLM batches:   0%|          | 0/42 [00:00<?, ?it/s]

In [25]:
from sklearn.linear_model import LogisticRegression

# sklearn's LogisticRegression as a 1D calibrator
calibrator = LogisticRegression(solver="lbfgs")

# Needs 2D input: reshape (-1,1)
calibrator.fit(minilm_val_probs_raw.reshape(-1, 1), y_val)

# Apply calibration to val and test
minilm_val_probs_cal = calibrator.predict_proba(minilm_val_probs_raw.reshape(-1, 1))[:, 1]
minilm_test_probs_cal = calibrator.predict_proba(minilm_test_probs_raw.reshape(-1, 1))[:, 1]


In [26]:
print("Raw MiniLM on test:")
print(compute_binary_metrics(y_test, minilm_test_probs_raw, threshold=0.5))

print("Calibrated MiniLM on test:")
print(compute_binary_metrics(y_test, minilm_test_probs_cal, threshold=0.5))


Raw MiniLM on test:
(0.8482475764354959, 0.3136593591905565, 0.20086393088552915, 0.7153846153846154, np.float64(0.8389106583072101))
Calibrated MiniLM on test:
(0.9515287099179717, 0.0, 0.0, 0.0, np.float64(0.8389106583072101))


# Ensemble Testing, Thresholds, Weights

In [27]:
minilm_probs = minilm_test_probs_cal

for w_nb, w_lstm, w_minilm in [
    (0.20, 0.70, 0.10),
    (0.30, 0.60, 0.10),
    (0.20, 0.60, 0.20),
]:
    ensemble_probs = w_nb * nb_probs + w_lstm * lstm_probs + w_minilm * minilm_probs
    thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

    rows = []
    for thr in thresholds:
        acc, f1, prec, rec, auc = compute_binary_metrics(y_test, ensemble_probs, threshold=thr)
        rows.append({
            "w_nb": w_nb,
            "w_lstm": w_lstm,
            "w_minilm": w_minilm,
            "threshold": thr,
            "Accuracy": acc,
            "F1-Score": f1,
            "Precision": prec,
            "Recall": rec,
            "ROC-AUC": auc,
        })

    df = pd.DataFrame(rows)
    print("\nWeights:", w_nb, w_lstm, w_minilm)
    display(df)



Weights: 0.2 0.7 0.1


,w_nb,w_lstm,w_minilm,threshold,Accuracy,F1-Score,Precision,Recall,ROC-AUC
0,0.2,0.7,0.1,0.30,0.982476,0.825279,0.798561,0.853846,0.982635
1,0.2,0.7,0.1,0.35,0.983221,0.827586,0.824427,0.830769,0.982635
2,0.2,0.7,0.1,0.40,0.984340,0.834646,0.854839,0.815385,0.982635
3,0.2,0.7,0.1,0.45,0.985086,0.837398,0.887931,0.792308,0.982635
4,0.2,0.7,0.1,0.50,0.985459,0.836820,0.917431,0.769231,0.982635



Weights: 0.3 0.6 0.1


,w_nb,w_lstm,w_minilm,threshold,Accuracy,F1-Score,Precision,Recall,ROC-AUC
0,0.3,0.6,0.1,0.30,0.983594,0.832061,0.825758,0.838462,0.981686
1,0.3,0.6,0.1,0.35,0.983967,0.832685,0.842520,0.823077,0.981686
2,0.3,0.6,0.1,0.40,0.985086,0.838710,0.881356,0.800000,0.981686
3,0.3,0.6,0.1,0.45,0.985459,0.836820,0.917431,0.769231,0.981686
4,0.3,0.6,0.1,0.50,0.984713,0.825532,0.923810,0.746154,0.981686



Weights: 0.2 0.6 0.2


,w_nb,w_lstm,w_minilm,threshold,Accuracy,F1-Score,Precision,Recall,ROC-AUC
0,0.2,0.6,0.2,0.30,0.983221,0.828897,0.819549,0.838462,0.981342
1,0.2,0.6,0.2,0.35,0.984340,0.834646,0.854839,0.815385,0.981342
2,0.2,0.6,0.2,0.40,0.984340,0.829268,0.879310,0.784615,0.981342
3,0.2,0.6,0.2,0.45,0.985459,0.835443,0.925234,0.761538,0.981342
4,0.2,0.6,0.2,0.50,0.985086,0.829060,0.932692,0.746154,0.981342


In [16]:
weight_sets = [
    (0.05, 0.45, 0.50),  # current
    (0.10, 0.40, 0.50),
    (0.05, 0.35, 0.60),
    (0.20, 0.40, 0.40),
    (0.10, 0.30, 0.60),
]

rows = []
for w_nb, w_lstm, w_minilm in weight_sets:
    ens_probs = w_nb * nb_probs + w_lstm * lstm_probs + w_minilm * minilm_probs
    acc, f1, prec, rec, auc = compute_binary_metrics(y_test, ens_probs, threshold=0.30)
    rows.append({
        "w_nb": w_nb,
        "w_lstm": w_lstm,
        "w_minilm": w_minilm,
        "Accuracy": acc,
        "F1-Score": f1,
        "Precision": prec,
        "Recall": rec,
        "ROC-AUC": auc,
    })

ensemble_weights_df = pd.DataFrame(rows)
display(ensemble_weights_df)


,w_nb,w_lstm,w_minilm,Accuracy,F1-Score,Precision,Recall,ROC-AUC
0,0.05,0.45,0.5,0.864653,0.389916,0.249462,0.892308,0.960571
1,0.10,0.40,0.5,0.865399,0.391231,0.250540,0.892308,0.957587
2,0.05,0.35,0.6,0.853468,0.367150,0.232179,0.876923,0.948433
3,0.20,0.40,0.4,0.875839,0.410619,0.266667,0.892308,0.969318
4,0.10,0.30,0.6,0.853468,0.367150,0.232179,0.876923,0.946980


# Common Fraud Signals from each model

## NB

In [29]:
import numpy as np

# Assuming: vectorizer (TfidfVectorizer), nb_model (MultinomialNB or similar)
feature_names = np.array(vectorizer.get_feature_names_out())

# Slice feature_log_prob_ to only include text features
# (The nb_model was trained on text features + 5 numeric features, making 5005 total)
fraud_log_probs_text = nb_model.feature_log_prob_[1, :len(feature_names)]  # class 1 = fraud
legit_log_probs_text = nb_model.feature_log_prob_[0, :len(feature_names)]

fraudiness = fraud_log_probs_text - legit_log_probs_text
top_idx = np.argsort(fraudiness)[-30:][::-1]  # top 30

top_nb_terms = feature_names[top_idx]
top_nb_scores = fraudiness[top_idx]

for term, score in zip(top_nb_terms, top_nb_scores):
    print(f"{term:20s}  {score:.3f}")

subsea                5.237
aker                  5.190
aker solution         5.142
business people       3.596
data entry            3.519
petroleum             3.451
typing                3.444
oil gas               3.423
clerk                 3.308
engineering design    3.234
work home             3.225
training provided     3.192
administrative assistant  3.081
hour day              3.012
home office           3.011
clerical              2.975
wage                  2.938
facilitating          2.930
earn                  2.924
referral bonus        2.887
encouraged            2.801
sensitive             2.799
signing               2.796
new exciting          2.784
tax free              2.774
position offer        2.769
figure                2.764
hired                 2.761
rn                    2.739
oil energy            2.729


- “subsea / Aker Solutions”: appears almost exclusively in a known fraud cluster.

-  “data entry”, “typing”, “work from home”, “training provided”: common in low‑effort, work‑from‑home scams.

-  “tax free”, “referral bonus”, “new exciting position offer”: strong “too good to be true” language.

## LSTM

In [30]:
high_risk_mask = lstm_probs >= 0.8  # or whatever high cutoff you like
high_risk_texts = X_test_text[high_risk_mask]


In [31]:
from collections import Counter
import re

def simple_tokens(text):
    return re.findall(r"[a-zA-Z]+", text.lower())

counter = Counter()
for t in high_risk_texts:
    counter.update(simple_tokens(t))

top_lstm_terms = counter.most_common(30)
for term, freq in top_lstm_terms:
    print(f"{term:20s}  {freq}")


work                  182
experience            170
project               159
service               155
skill                 139
customer              136
engineering           123
product               122
amp                   119
position              118
team                  117
company               112
u                     103
level                 91
process               90
entry                 89
management            88
business              81
year                  80
oil                   77
industry              76
data                  74
new                   73
system                73
ability               70
solution              69
communication         69
test                  68
job                   66
candidate             65


-  When we look only at jobs our LSTM is very confident are fraudulent, we see lots of generic job‑search language (‘work’, ‘experience’, ‘position’, ‘team’), which confirms that scammers mimic normal job postings.

-  Within that high‑risk slice, we also see heavier representation of certain patterns: entry‑level customer/data roles and specific industry clusters (oil/energy/engineering), which match real‑world reports of scam campaigns in those areas.”



## MINILM

In [41]:
from collections import Counter
import numpy as np

important_tokens = Counter()

# Try several high-risk thresholds and pick the first with enough examples
candidate_thresholds = [0.7, 0.6, 0.5]
min_examples = 50  # want at least this many texts

selected_indices = None
chosen_threshold = None

for thr in candidate_thresholds:
    idx = np.where(minilm_probs >= thr)[0]  # use calibrated MiniLM probs
    print(f"Threshold {thr}: {len(idx)} high-risk examples")
    if len(idx) >= min_examples and chosen_threshold is None:
        chosen_threshold = thr
        selected_indices = idx

if selected_indices is None:
    # Fallback: take top-N by probability if even 0.5 is too strict
    N = min(100, len(minilm_probs))
    selected_indices = np.argsort(minilm_probs)[-N:]
    chosen_threshold = "top-N"

print("\nUsing threshold:", chosen_threshold, "with", len(selected_indices), "examples\n")

# Aggregate IG tokens over selected high-risk examples
for idx in selected_indices:
    text = X_text_test_nb[idx]
    token_scores = explain_minilm_with_ig(text)  # real IG function from 07_minilm_IG

    # Sort tokens by absolute attribution magnitude and keep top 10
    top_tokens = [
        tok for tok, score in sorted(token_scores, key=lambda x: -abs(x[1]))[:10]
    ]
    important_tokens.update(top_tokens)

print("Top MiniLM+IG fraud tokens:\n")
for tok, freq in important_tokens.most_common(30):
    print(f"{tok:20s}  {freq}")


Threshold 0.7: 0 high-risk examples
Threshold 0.6: 0 high-risk examples
Threshold 0.5: 0 high-risk examples

Using threshold: top-N with 100 examples

Top MiniLM+IG fraud tokens:

u                     86
position              35
full                  28
provide               28
time                  26
assistant             24
permanent             24
entry                 20
administrative        19
data                  17
service               14
oh                    14
manager               13
clerk                 12
tx                    11
ca                    10
cleveland             10
home                  10
payroll               10
based                 9
job                   8
senior                8
customer              7
sale                  7
maintenance           7
houston               6
fl                    6
mo                    6
louis                 6
engineer              6


-  Very frequent, generic job words: u, position, full, provide, time, assistant, entry, administrative, data, service, manager, clerk, customer, job, engineer.

-  Location/state abbreviations: tx, ca, oh, fl, mo, cleveland, houston, louis.

-  Home/office‑type language: home, based, payroll, permanent, maintenance.



-  NB: “These phrases are statistically much more common in fraud posts (e.g., ‘work home’, ‘tax free’, ‘referral bonus’, ‘data entry’).”

-  LSTM: “These words frequently appear in posts our neural network is very confident are fraud (e.g., ‘entry’, ‘data’, ‘oil’, ‘industry’).”

-  MiniLM+IG: “These tokens are repeatedly highlighted by Integrated Gradients as driving MiniLM’s fraud predictions (e.g., ‘full time position’, ‘administrative assistant’, ‘data entry clerk’, ‘home‑based’, plus certain U.S. locations).”

